# LAB 4 — Arquiteturas Analíticas Modernas e MLOps

**Disciplina** Engenharia de Dados para IA

**Aula** 4 — Arquiteturas Modernas de Dados

**Duração do Lab** ~1h30 (em aula)

> **Considerar apenas os itens 1.2** (Posicionamento Estratégico da LODLog) **e 1.3** (Relacional vs. Não-relacional). As demais atividades são de apoio e discussão.

## Objetivos de Aprendizagem

Ao final deste laboratório, o grupo será capaz de:

1. Diferenciar as características, vantagens e limitações de **Data Warehouse**, **Data Lake**, **Data Lakehouse** e **Data Mesh**.
2. Avaliar um cenário de negócio de logística e posicioná-lo na arquitetura analítica mais adequada ao seu momento de crescimento.
3. Projetar um pipeline de dados distribuído utilizando a **Arquitetura Medallion** (Bronze, Silver e Gold), mapeando o fluxo de processamento Batch e Streaming.
4. Estruturar o papel da Engenharia de Dados no ciclo de vida de modelos preditivos (**MLOps**), especificando a integração com Feature Stores e mecanismos de detecção de data drift.
5. Elaborar um **Roadmap de Evolução Arquitetural** e um checklist de conformidade com a **LGPD**.

## Parte 0 — Contexto (5 min)

Nos laboratórios anteriores (LABs 1 a 3), vocês realizaram a modelagem lógica e física do banco analítico da LODLog e construíram views para preparação de modelos de ML no Databricks. Agora, o desafio é expandir esse projeto para uma visão integrada de arquitetura de dados moderna. Vocês deverão desenhar a solução técnica corporativa da LODLog que suporte:

* **Big Data:** crescimento exponencial do volume de entregas e dados de sensores IoT.
* **Variedade de Fontes:** arquivos JSON de e-commerce, bancos relacionais ERP, APIs de clima, logs de telemetria da frota e posts públicos de redes sociais (X) sobre greves, bloqueios e acidentes.
* **Modelos de ML em Produção (MLOps):** suporte estável para os modelos de *Predição de Atrasos* e *Manutenção Preditiva*.
* **Governança de Dados:** controle de acesso, auditoria e conformidade regulatória (LGPD).

## Parte 1 — Arquiteturas Analíticas: O Quadrante de Decisão (20 min)

### Atividade 1.1: Complete o Quadrante

Preencha os espaços em branco na tabela comparativa de arquiteturas analíticas modernas. Use esta tabela como base conceitual para o relatório de entrega.

| Arquitetura | Tipo de Dados Suportados | Estratégia de Schema | Caso Ideal de Uso (Melhor para) | Limitações (Evitar quando) |
| --- | --- | --- | --- | --- |
| **Data Warehouse** | Estruturado apenas (tabelas relacionais) | Schema-on-write (validação rígida na escrita) | Business Intelligence tradicional, relatórios de auditoria e dashboards executivos estáveis. | Dados semiestruturados de alta frequência (IoT), mídias não estruturadas e ML exploratório. |
| **Data Lake** | ____________________ | ____________________ | Armazenamento em escala de dados brutos, Big Data e exploração de Data Science. | Usuários finais de negócio necessitam de consultas rápidas e diretas (risco de *Data Swamp*). |
| **Data Lakehouse** | Estruturado, semiestruturado e não estruturado | Híbrido (metadados abertos com ACID) | ____________________ | Projetos muito pequenos com equipes sem maturidade ou sem demandas de IA. |
| **Data Mesh** | ____________________ | Descentralizado (por domínio de negócio) | Grandes organizações com múltiplos domínios de negócio autônomos. | ____________________ |

### Atividade 1.2: Posicionamento Estratégico da LODLog

Discutam o seguinte cenário de crescimento e tomem uma decisão de arquitetura para a LODLog:

* **Momento Atual:** 450 veículos na frota, 5 centros de distribuição ativos, volume de ~50 mil entregas por mês.
* **Projeção para daqui a 3 anos:** 1.500 veículos na frota, 30 centros de distribuição ativos, volume projetado de ~250 mil entregas por mês, integração em tempo real de dados de telemetria (IoT) e operação em 3 países da América Latina (iniciando fusões e aquisições).

**Questão:** *"Qual arquitetura analítica vocês recomendam para a LODLog hoje? Qual a evolução proposta para os próximos 3 anos? Justifiquem a escolha em 6 a 10 linhas no relatório final."*

**Resposta do grupo:**

### Atividade 1.3: Relacional vs. Não-relacional — sinais públicos do X (20 min, em grupo)

Protestos, greves, bloqueios de rodovias e portos, acidentes e congestionamentos atrasam entregas. Esses eventos costumam aparecer primeiro em **posts públicos no X**. Se a LODLog coletar esses posts, pode **re-roteirizar**, **antecipar atrasos** e **ajustar janelas de entrega** antes que o cliente reclame.

A coleta real usaria a busca recente da API do X (`GET /2/tweets/search/recent`), que devolve JSON. Para o LAB usamos **10 posts fictícios** no formato (simplificado) da API v2. Para coletar posts reais com a sua própria conta de desenvolvedor, há o notebook **opcional** `LAB4-X-API.ipynb` (tem custo por post lido). Veja um deles:

```json
{
  "data": {
    "id": "1838000000000000001",
    "text": "Caminhoneiros bloqueiam a Anhanguera (SP-330) no km 92 sentido capital. Fila de 8 km. #bloqueio #greve",
    "created_at": "2026-09-14T09:12:00.000Z",
    "author_id": "9001",
    "lang": "pt",
    "public_metrics": {
      "retweet_count": 340,
      "like_count": 512
    },
    "entities": {
      "hashtags": [
        {
          "tag": "bloqueio"
        },
        {
          "tag": "greve"
        }
      ]
    },
    "geo": {
      "place_id": "pl_cps"
    }
  },
  "includes": {
    "users": [
      {
        "id": "9001",
        "username": "transito_cps"
      }
    ],
    "places": [
      {
        "id": "pl_cps",
        "full_name": "Campinas, Brasil"
      }
    ]
  }
}
```

Repare que nem todo post tem os mesmos campos: alguns não têm localização (`geo`/`places`), outros não têm hashtags, e retweets trazem `referenced_tweets`. As células abaixo usam as tabelas da LODLog carregadas pelo seed (LAB 2/3).

**1. Bronze — guardar o JSON como chegou (schema-on-read):**

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS lodlog_lake;

CREATE OR REPLACE TABLE lodlog_lake.x_posts_bronze USING DELTA AS
SELECT payload, current_timestamp() AS ingerido_em
FROM VALUES
  ('{"data": {"id": "1838000000000000001", "text": "Caminhoneiros bloqueiam a Anhanguera (SP-330) no km 92 sentido capital. Fila de 8 km. #bloqueio #greve", "created_at": "2026-09-14T09:12:00.000Z", "author_id": "9001", "lang": "pt", "public_metrics": {"retweet_count": 340, "like_count": 512}, "entities": {"hashtags": [{"tag": "bloqueio"}, {"tag": "greve"}]}, "geo": {"place_id": "pl_cps"}}, "includes": {"users": [{"id": "9001", "username": "transito_cps"}], "places": [{"id": "pl_cps", "full_name": "Campinas, Brasil"}]}}'),
  ('{"data": {"id": "1838000000000000002", "text": "RT @transito_cps: Caminhoneiros bloqueiam a Anhanguera (SP-330) no km 92 sentido capital. Fila de 8 km. #bloqueio #greve", "created_at": "2026-09-14T09:20:00.000Z", "author_id": "9002", "lang": "pt", "public_metrics": {"retweet_count": 0, "like_count": 0}, "entities": {"hashtags": [{"tag": "bloqueio"}, {"tag": "greve"}]}, "referenced_tweets": [{"type": "retweeted", "id": "1838000000000000001"}]}, "includes": {"users": [{"id": "9002", "username": "motorista_ana"}]}}'),
  ('{"data": {"id": "1838000000000000003", "text": "Marginal Tietê parada na altura da Ponte do Limão: acidente com carreta envolvendo 3 veículos. Evitem a região.", "created_at": "2026-09-16T20:40:00.000Z", "author_id": "9003", "lang": "pt", "public_metrics": {"retweet_count": 125, "like_count": 230}, "geo": {"place_id": "pl_sp"}}, "includes": {"users": [{"id": "9003", "username": "radar_sp"}], "places": [{"id": "pl_sp", "full_name": "São Paulo, Brasil"}]}}'),
  ('{"data": {"id": "1838000000000000004", "text": "Greve dos portuários em Paranaguá entra no 2º dia, navios aguardando atracação. Cargas para Curitiba atrasadas #greveportuaria", "created_at": "2026-09-17T12:05:00.000Z", "author_id": "9004", "lang": "pt", "public_metrics": {"retweet_count": 88, "like_count": 140}, "entities": {"hashtags": [{"tag": "greveportuaria"}]}}, "includes": {"users": [{"id": "9004", "username": "portos_br"}]}}'),
  ('{"data": {"id": "1838000000000000005", "text": "Protesto fecha o Anel Rodoviário nos dois sentidos desde as 6h. #protesto", "created_at": "2026-09-21T10:30:00.000Z", "author_id": "9005", "lang": "pt", "public_metrics": {"retweet_count": 410, "like_count": 700}, "entities": {"hashtags": [{"tag": "protesto"}]}, "geo": {"place_id": "pl_bh"}}, "includes": {"users": [{"id": "9005", "username": "bh_agora"}], "places": [{"id": "pl_bh", "full_name": "Belo Horizonte, Brasil"}]}}'),
  ('{"data": {"id": "1838000000000000006", "text": "Congestionamento de 12 km na Freeway (BR-290) chegando em Porto Alegre por causa da chuva forte", "created_at": "2026-09-22T21:10:00.000Z", "author_id": "9006", "lang": "pt", "public_metrics": {"retweet_count": 64, "like_count": 98}, "geo": {"place_id": "pl_poa"}}, "includes": {"users": [{"id": "9006", "username": "gaucha_transito"}], "places": [{"id": "pl_poa", "full_name": "Porto Alegre, Brasil"}]}}'),
  ('{"data": {"id": "1838000000000000007", "text": "Alguém recomenda restaurante bom perto da Paulista? #fome", "created_at": "2026-09-23T15:00:00.000Z", "author_id": "9007", "lang": "pt", "public_metrics": {"retweet_count": 1, "like_count": 3}, "entities": {"hashtags": [{"tag": "fome"}]}, "geo": {"place_id": "pl_sp"}}, "includes": {"users": [{"id": "9007", "username": "joao_paulista"}], "places": [{"id": "pl_sp", "full_name": "São Paulo, Brasil"}]}}'),
  ('{"data": {"id": "1838000000000000008", "text": "Tombamento de caminhão na BR-116 (Régis Bittencourt) perto de Curitiba, pista interditada sentido SP", "created_at": "2026-09-24T08:50:00.000Z", "author_id": "9008", "lang": "pt", "public_metrics": {"retweet_count": 230, "like_count": 310}, "geo": {"place_id": "pl_cwb"}}, "includes": {"users": [{"id": "9008", "username": "prf_pr_info"}], "places": [{"id": "pl_cwb", "full_name": "Curitiba, Brasil"}]}}'),
  ('{"data": {"id": "1838000000000000009", "text": "Lentidão forte na Bandeirantes entre Campinas e Jundiaí, obras na pista #transito", "created_at": "2026-09-28T11:15:00.000Z", "author_id": "9001", "lang": "pt", "public_metrics": {"retweet_count": 45, "like_count": 80}, "entities": {"hashtags": [{"tag": "transito"}]}, "geo": {"place_id": "pl_cps"}}, "includes": {"users": [{"id": "9001", "username": "transito_cps"}], "places": [{"id": "pl_cps", "full_name": "Campinas, Brasil"}]}}'),
  ('{"data": {"id": "1838000000000000010", "text": "Greve geral convocada para amanhã em todo o país", "created_at": "2026-09-29T13:00:00.000Z", "author_id": "9009", "lang": "pt", "public_metrics": {"retweet_count": 1500, "like_count": 2100}}, "includes": {"users": [{"id": "9009", "username": "sindicato_geral"}]}}')
AS t(payload);

**2. Explorar o JSON com SQL** — cada campo é lido só na hora da consulta. Quais colunas vêm vazias em alguns posts? Como você modelaria isso em tabelas relacionais?

In [ ]:
%sql
SELECT get_json_object(payload, '$.data.id')                     AS post_id,
       get_json_object(payload, '$.data.created_at')             AS criado_em_utc,
       get_json_object(payload, '$.includes.places[0].full_name') AS local,
       get_json_object(payload, '$.data.entities.hashtags')      AS hashtags,
       get_json_object(payload, '$.data.referenced_tweets[0].type') AS referencia,
       get_json_object(payload, '$.data.text')                   AS texto
FROM lodlog_lake.x_posts_bronze;

**3. Silver — do post ao evento estruturado.** Extraímos data (no fuso de Brasília), cidade e tipo de evento; descartamos retweets (duplicatas) e **não levamos o autor** (LGPD: minimização).

In [ ]:
%sql
CREATE OR REPLACE TABLE lodlog_lake.x_eventos_silver USING DELTA AS
WITH posts AS (
  SELECT get_json_object(payload, '$.data.id') AS post_id,
         to_date(from_utc_timestamp(
           CAST(get_json_object(payload, '$.data.created_at') AS TIMESTAMP),
           'America/Sao_Paulo')) AS data_evento,
         lower(get_json_object(payload, '$.data.text')) AS texto,
         split(get_json_object(payload, '$.includes.places[0].full_name'), ',')[0] AS cidade_place,
         get_json_object(payload, '$.data.referenced_tweets[0].type') AS referencia
  FROM lodlog_lake.x_posts_bronze
)
SELECT post_id,
       data_evento,
       COALESCE(cidade_place,
                CASE WHEN texto LIKE '%campinas%'       THEN 'Campinas'
                     WHEN texto LIKE '%são paulo%'      THEN 'São Paulo'
                     WHEN texto LIKE '%belo horizonte%' THEN 'Belo Horizonte'
                     WHEN texto LIKE '%curitiba%'       THEN 'Curitiba'
                     WHEN texto LIKE '%porto alegre%'   THEN 'Porto Alegre'
                END) AS cidade,
       CASE WHEN texto RLIKE 'bloque|protesto|manifesta'       THEN 'Bloqueio/Protesto'
            WHEN texto RLIKE 'greve|paralisa'                  THEN 'Greve'
            WHEN texto RLIKE 'acidente|tombamento|colis'       THEN 'Acidente'
            WHEN texto RLIKE 'congestionamento|lentid|parada'  THEN 'Congestionamento'
            ELSE 'Irrelevante'
       END AS tipo_evento
FROM posts
WHERE referencia IS NULL OR referencia <> 'retweeted';

SELECT * FROM lodlog_lake.x_eventos_silver ORDER BY data_evento;

**4. Juntar o não-relacional com o relacional:** eventos (Silver) × entregas do CD da mesma cidade no mesmo dia (`fato_entregas` + `dim_centro_distribuicao`), comparadas com a taxa de atraso normal do CD.

In [ ]:
%sql
WITH eventos AS (
  SELECT data_evento, cidade, concat_ws(', ', collect_set(tipo_evento)) AS eventos
  FROM lodlog_lake.x_eventos_silver
  WHERE tipo_evento <> 'Irrelevante' AND cidade IS NOT NULL
  GROUP BY data_evento, cidade
),
normal_cd AS (
  SELECT sk_cd_origem, ROUND(100 * AVG(indicador_atraso), 1) AS pct_atraso_normal
  FROM lodlog_dw.fato_entregas
  GROUP BY sk_cd_origem
)
SELECT e.data_evento, e.cidade, e.eventos, cd.codigo AS cd,
       COUNT(fe.sk_entrega)                    AS entregas_no_dia,
       ROUND(100 * AVG(fe.indicador_atraso), 1) AS pct_atraso_no_dia,
       n.pct_atraso_normal
FROM eventos e
JOIN lodlog_dw.dim_centro_distribuicao cd ON cd.municipio = e.cidade
JOIN normal_cd n                          ON n.sk_cd_origem = cd.sk_cd
LEFT JOIN lodlog_dw.fato_entregas fe      ON fe.sk_cd_origem = cd.sk_cd
                                         AND fe.data_entrega = e.data_evento
GROUP BY e.data_evento, e.cidade, e.eventos, cd.codigo, n.pct_atraso_normal
ORDER BY e.data_evento;

**Responda (grupo):**

1. Para cada dado da LODLog, indique se você guardaria em banco **relacional** ou **não-relacional** e, se não-relacional, de qual tipo (documento, chave-valor, colunar/série temporal, grafo, busca textual). Justifique em uma frase.

| Dado | Relacional ou não-relacional? | Tipo | Por quê? |
| --- | --- | --- | --- |
| Entregas e pedidos (`fato_entregas`, `pedido`) |  |  |  |
| Cadastro de clientes e contratos |  |  |  |
| Posts públicos do X |  |  |  |
| Telemetria IoT da frota (GPS a cada segundo) |  |  |  |
| Cache do tempo estimado de rota por região (consulta em milissegundos) |  |  |  |

2. Por que guardamos o JSON bruto na Bronze em vez de criar direto uma tabela relacional a partir da API? O que acontece se o X acrescentar ou remover campos da resposta?
3. A busca recente da API do X cobre só os **últimos 7 dias** e é **cobrada por leitura**. Que latência de coleta (ver Atividade 2.2) e que filtros de busca vocês usariam para controlar o custo?
4. Os dados de entrega do seed são sintéticos e não têm relação real com os posts. Em produção, como esse cruzamento viraria uma **feature** do modelo de atraso (Parte 3)? Por exemplo: `evento_na_cidade_do_cd_no_dia` (0/1).
5. A classificação por palavras-chave errou ou deixou algo de fora? (Veja o post da "greve geral" sem cidade e o post irrelevante.) Que riscos isso traz e como vocês melhorariam?
6. LGPD: que campos da resposta da API **não** devem chegar à Silver/Gold? Por quê?

## Parte 2 — Camadas Medallion na Prática (30 min)

### Atividade 2.1: Mapeamento da Arquitetura

Para o cenário da LODLog, a empresa utiliza quatro fontes principais de dados. Preencham a tabela definindo quais transformações, limpezas e filtros os dados sofrerão ao longo das três camadas da Arquitetura Medallion:

| Fonte Original | Zonas de Ingestão / Camada Bronze | Camada Silver (Limpeza e Conformidade) | Camada Gold (Consumo e KPIs) |
| --- | --- | --- | --- |
| **Telemetria IoT (GPS e Sensores da Frota)** | Dados em JSON bruto salvos em diretórios particionados por data. | Schema aplicado, validação de coordenadas geográficas e filtragem de duplicatas de sensores. | ____________________ |
| **Pedidos do TMS (Banco Relacional)** | Tabelas operacionais replicadas na Bronze sem transformações. | ____________________ | Métricas agregadas de frete por CD, taxa de sinistros e custos operacionais. |
| **Estoque do WMS (Arquivos CSV)** | ____________________ | Limpeza de registros nulos e uniformização de SKUs com o cadastro de produtos. | Nível de giro de estoque por CD e alertas de ruptura. |
| **APIs Externas de Clima (JSON)** | Retorno da API armazenado com timestamp de coleta. | Conversão de temperaturas e padronização das condições de chuva/sol. | ____________________ |

### Atividade 2.2: Latência de Processamento

Definam a latência de processamento adequada (**Batch**, **Real-Time/Streaming** ou **Híbrido**) para os seguintes fluxos de dados, justificando a decisão com base nas necessidades reais da operação da LODLog:

| Fluxo de Dados | Latência Proposta | Justificativa de Negócio / Decisão Operacional |
| --- | --- | --- |
| **Ingestão IoT da Frota → Bronze** | Streaming (Kinesis/Kafka) | Necessário para saber a localização dos motoristas em tempo real e monitorar a integridade das cargas críticas. |
| **Carga do ERP → Camada Silver** | ____________________ | ____________________ |
| **Cálculo do SLA de Entregas → Dashboard Executivo** | ____________________ | ____________________ |
| **Detecção de Falha Mecânica IoT → Alerta no TMS** | ____________________ | ____________________ |

### Atividade 2.3: Desenho Técnico do Pipeline (Exemplo de arquitetura de ingestão e armazenamento)

Esbocem ou diagramem o pipeline completo da LODLog, desde as origens até os consumidores finais (Dashboards de BI e Modelos de ML). Tomem como base um **exemplo de arquitetura de ingestão e armazenamento** em nuvem. Indiquem explicitamente tecnologias sugeridas que englobem conceitos além da AWS, integrando obrigatoriamente os **conceitos do Databricks Lakehouse**, tais como:

* Armazenamento resiliente e escalável no Data Lake usando cloud object storage (ex: AWS S3, Azure ADLS ou GCP Cloud Storage).
* Adoção da camada **Delta Lake** sobre o storage para conferir confiabilidade de DW (transações ACID, versionamento).
* Ingestão e processamento de dados massivos e unificados (batch e streaming) com **Apache Spark** (computação).
* Organização do ciclo de vida dos dados usando a arquitetura **Medallion** (camadas Bronze, Silver e Gold).

## Parte 3 — MLOps e Engenharia de Dados (25 min)

### Atividade 3.1: O Ciclo de Vida do Dado no MLOps (10 min, individual)

Analise o diagrama do pipeline de Machine Learning abaixo e responda:

```
[Dados Brutos] --(1)--> [Bronze] --(2)--> [Silver] --(3)--> [Gold / Feature Store] --(4)--> [Treinamento ML] --(5)--> [Inferência em Prod]
```

**Perguntas:**

1. Qual a responsabilidade técnica do **Engenheiro de Dados** nos passos (1), (2) e (3)? E qual a responsabilidade do **Cientista de Dados** no passo (4)?
2. Em produção, o modelo consome dados novos na etapa (5). Por que o pipeline de dados que gera as features para a inferência em tempo real deve ser **idêntico** ao pipeline que gerou os dados de treinamento no passo (3)?

### Atividade 3.2: Desenho do Pipeline de MLOps (15 min, em grupo)

Escolha **um dos dois modelos preditivos** da LODLog:

* **Modelo A: Predição de Atraso de Entregas** (Classificação — prevê se a viagem atrasará).
* **Modelo B: Manutenção Preditiva dos Veículos** (Séries Temporais — prevê falha de componentes mecânicos com base em telemetria IoT).

Para o modelo escolhido, descreva o pipeline de MLOps respondendo de forma objetiva aos seguintes tópicos:

1. **Origem de Treinamento:** de qual tabela ou view da camada Gold o modelo lerá os dados de treinamento?
2. **Frequência de Atualização:** com qual frequência os dados da Feature Store devem ser atualizados e o que dispara o retreinamento do modelo?
3. **Inferência:** a predição em produção deve rodar em Batch (ex: toda noite para as viagens do dia seguinte) ou Real-Time (ex: atualizar a probabilidade a cada nova coordenada de GPS)? Justifiquem a escolha.
4. **Monitoramento de Data Drift:** como a engenharia de dados pode monitorar se a distribuição dos dados em produção está mudando (ex: novos motoristas contratados com perfil diferente) e como acionar alertas de degradação do modelo?

## Parte 4 — Governança de Dados, LGPD e Roadmap (20 min)

### Atividade 4.1: Checklist de Conformidade com a LGPD (10 min, em grupo)

Preencham o checklist descrevendo como o pipeline de dados da LODLog garantirá a conformidade com a LGPD nos pontos críticos de engenharia:

| Exigência da LGPD | Impacto no Pipeline da LODLog | Como Implementar Tecnicamente no Data Lakehouse? |
| --- | --- | --- |
| **Anonimização de Dados Pessoais** | Dados de motoristas (CNH, CPF) e clientes (nome, endereço de entrega). | Mascaramento dinâmico de colunas na camada Silver ou criação de views anonimizadas na Gold utilizando hashes/criptografia. |
| **Direito de Exclusão (Esquecimento)** | O cliente solicita a remoção completa do seu histórico pessoal do banco de dados. | ____________________ |
| **Minimização e Retenção de Dados** | Armazenamento de logs IoT indefinidamente gera altos custos e riscos legais. | ____________________ |
| **Controle de Acesso e Auditoria** | Prevenir que analistas de BI visualizem dados salariais de motoristas. | ____________________ |

### Atividade 4.2: Roadmap de Implementação

Estruturem um plano realista de implementação de 18 meses para a LODLog, dividindo o roadmap técnico nas três fases clássicas de evolução de dados:

| Fase de Evolução | Período | Objetivo Principal de Negócio | Arquitetura Alvo | Principais Tecnologias do Bloco |
| --- | --- | --- | --- | --- |
| **Fase 1 (MVP e Fundações)** | Meses 1 a 3 | Unificar as fontes operacionais do WMS, TMS e ERP em repositório central. | Data Lake / DW inicial | Databricks SQL, S3, Delta Lake. |
| **Fase 2 (Escala e Qualidade)** | Meses 4 a 9 | ____________________ | Data Lakehouse completo | ____________________ |
| **Fase 3 (IA Nativa e MLOps)** | Meses 10 a 18 | ____________________ | ____________________ | ____________________ |

---

**Dica do Professor:** Arquitetura de dados não é uma escolha dogmática ou apenas sobre seguir o *hype* do mercado. O papel do gestor de tecnologia é encontrar a melhor solução custo-benefício para o momento de maturidade da empresa. Justifiquem cada tecnologia sugerida com base no valor de negócio que ela desbloqueia e na complexidade que a equipe consegue operar. Bom trabalho!